# Motor de Linkage — Notebook de validação (backend)

Objetivo deste notebook: rodar o `MotorDeLinkage` de ponta a ponta com
dados **sintéticos** (não são BOs reais), pra você validar a integração
antes de ligar em dados de verdade.

Pré-requisitos:
- `motor_linkage.py` precisa estar na mesma pasta deste notebook.
- Este notebook usa o encoder **real** de produção
  (`sentence-transformers` + BERTimbau) -- na primeira execução, ele
  baixa o modelo da internet (alguns minutos, dependendo da conexão).
  Se estiver rodando num ambiente sem internet, veja a nota na célula
  do encoder sobre o fallback offline (TF-IDF), útil só pra testar a
  mecânica do pipeline (não reflete a qualidade real do sistema).
- O classificador usado aqui é o **fallback** (`clf=None`, média das
  features) — se você já tiver o `.joblib` treinado (`modelo_v1_classificador.joblib`),
  troque na célula de instanciação do motor, conforme indicado.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd "/content/drive/MyDrive/FACEPE-PCPE/BOs - Exemplo/Linkage/Código/09-07/final/backend"

/content/drive/.shortcut-targets-by-id/1g1aqpfQi2mUtv9S0flNQQZM9m1oz4PCU/FACEPE-PCPE/BOs - Exemplo/Linkage/Código/09-07/final/backend


In [4]:
import sys, os
#sys.path.insert(0, ".")  # ajuste se motor_linkage.py estiver em outra pasta

from motor_linkage import MotorDeLinkage
import numpy as np

## 1. Encoder — versão de produção (BERTimbau via sentence-transformers)

Requer `pip install sentence-transformers`. Na primeira execução baixa
o modelo (~400MB) -- depois fica em cache local.

In [5]:
from sentence_transformers import SentenceTransformer

print("Carregando encoder (primeira vez baixa o modelo, pode demorar)...")
encoder = SentenceTransformer("neuralmind/bert-base-portuguese-cased")
print("Encoder carregado. Dimensão do embedding:", encoder.get_sentence_embedding_dimension())

# Alternativa mais leve, já treinada especificamente pra similaridade de
# sentença (não só token) -- costuma performar melhor nesse tipo de tarefa:
# encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Fallback offline (SEM sentence-transformers/internet) -- só pra testar
# a mecânica do pipeline, não reflete a qualidade real do sistema:
#
# from sklearn.feature_extraction.text import TfidfVectorizer
# class TfidfEncoderAdapter:
#     def __init__(self, textos_referencia):
#         self.vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95)
#         self.vec.fit(textos_referencia)
#     def encode(self, textos, convert_to_numpy=True):
#         return self.vec.transform(textos).toarray()
# encoder = TfidfEncoderAdapter([b["texto"] for b in bos_sinteticos])

Carregando encoder (primeira vez baixa o modelo, pode demorar)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoder carregado. Dimensão do embedding: 768


/tmp/ipykernel_1166/3228224506.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Encoder carregado. Dimensão do embedding:", encoder.get_sentence_embedding_dimension())


## 2. Dados sintéticos de exemplo

8 BOs fictícios, desenhados de propósito pra você ver os 3 casos de uso
em ação:
- `BO-001` e `BO-002`: mesma placa (deve conectar no caso de uso 1)
- `BO-003` e `BO-004`: mesma pessoa, nome digitado com pequena variação
  (deve conectar no caso de uso 3 -- fuzzy match)
- `BO-005`, `BO-006`, `BO-007`: mesma natureza ("furto"), datas/locais
  próximos (deve formar cluster no caso de uso 2)
- `BO-008`: sem conexão com nenhum outro (caso de controle)

In [6]:
geo_lookup = {
    "RECIFE": (-8.0476, -34.8770),
    "OLINDA": (-8.0089, -34.8553),
    "JABOATAO": (-8.1130, -35.0148),
}

bos_sinteticos = [
    {
        "id": "BO-001",
        "texto": "Furto de veiculo placa ABC1D23 na Av. Boa Viagem, Recife.",
        "data_fato": "10/07/2026",
        "municipio": "RECIFE",
        "natureza": "furto",
        "entidades": [{"texto": "ABC1D23", "tipo": "PLACA"}],
    },
    {
        "id": "BO-002",
        "texto": "Veiculo de placa ABC-1D23 encontrado abandonado em Olinda.",
        "data_fato": "12/07/2026",
        "municipio": "OLINDA",
        "natureza": "achado e perdido",
        "entidades": [{"texto": "ABC-1D23", "tipo": "PLACA"}],
    },
    {
        "id": "BO-003",
        "texto": "Vitima Joao da Silva relata golpe financeiro via WhatsApp.",
        "data_fato": "01/07/2026",
        "municipio": "RECIFE",
        "natureza": "estelionato",
        "entidades": [{"texto": "Joao da Silva", "tipo": "PESSOA_NOME"}],
    },
    {
        "id": "BO-004",
        "texto": "Novo boletim complementar de JOAO  DA  SILVA sobre o mesmo golpe.",
        "data_fato": "03/07/2026",
        "municipio": "RECIFE",
        "natureza": "estelionato",
        "entidades": [{"texto": "JOAO  DA  SILVA", "tipo": "PESSOA_NOME"}],
    },
    {
        "id": "BO-005",
        "texto": "Furto em residencia no bairro de Boa Vista, Recife.",
        "data_fato": "05/07/2026",
        "municipio": "RECIFE",
        "natureza": "furto",
        "entidades": [{"texto": "furto", "tipo": "OCORRENCIA"},
                       {"texto": "Boa Vista", "tipo": "LOCAL_AREA"}],
    },
    {
        "id": "BO-006",
        "texto": "Furto qualificado em imovel no bairro de Boa Vista.",
        "data_fato": "07/07/2026",
        "municipio": "RECIFE",
        "natureza": "furto qualificado",
        "entidades": [{"texto": "furto qualificado", "tipo": "OCORRENCIA"},
                       {"texto": "Boa Vista", "tipo": "LOCAL_AREA"}],
    },
    {
        "id": "BO-007",
        "texto": "Furto de bens em casa proximo a Boa Vista, mesma regiao.",
        "data_fato": "06/07/2026",
        "municipio": "RECIFE",
        "natureza": "furto",
        "entidades": [{"texto": "furto", "tipo": "OCORRENCIA"},
                       {"texto": "Boa Vista", "tipo": "LOCAL_AREA"}],
    },
    {
        "id": "BO-008",
        "texto": "Perturbacao do sossego em bar na Boa Viagem, sem outras informacoes.",
        "data_fato": "20/07/2026",
        "municipio": "RECIFE",
        "natureza": "perturbacao do sossego",
        "entidades": [{"texto": "perturbacao do sossego", "tipo": "OCORRENCIA"}],
    },
]

print(f"{len(bos_sinteticos)} BOs sintéticos prontos.")

8 BOs sintéticos prontos.


## 3. Instanciar o motor e carregar os BOs

In [7]:
# clf=None -> usa o fallback (média das features). Se você já tiver o
# .joblib treinado, troque por:
import joblib
clf = joblib.load("modelo_v1_classificador.joblib")
#   motor = MotorDeLinkage(encoder, geo_lookup, clf=clf, ...)
motor = MotorDeLinkage(encoder, geo_lookup, clf=None,
                        tau_geo_km=15.0, tau_dias=30.0,
                        limiar_descricao=0.6)

for bo in bos_sinteticos:
    motor.adicionar_bo(bo, entidades=bo["entidades"])

print(f"BOs carregados no índice: {len(motor.ids)}")

BOs carregados no índice: 8


## 4. Caso de uso 1 — "Dado esse BO, quais estão conectados a ele?"

Esperado: `BO-001` deve aparecer conectado a `BO-002` (mesma placa).

In [9]:
acima_threshold, top_k = motor.bos_conectados("BO-001", k=5, threshold=0.6)

print("Acima do threshold:")
for r in acima_threshold:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

print("\nTop 5 (independente de threshold):")
for r in top_k:
    print(f"  {r['bo_id']}: score={r['score']:.3f}")

Acima do threshold:
  BO-002: score=0.643 | motivo: mesmo(a) PLACA

Top 5 (independente de threshold):
  BO-002: score=0.643
  BO-005: score=0.333
  BO-007: score=0.321
  BO-006: score=0.320
  BO-008: score=0.276


## 5. Caso de uso 2 — "Dado esse tipo de crime, quais grupos se conectam?"

Esperado: `BO-005`, `BO-006`, `BO-007` (todos "furto", mesmo local
`LOCAL_AREA=Boa Vista`, datas próximas) devem cair no mesmo cluster --
o overlap de local exato garante isso mesmo com o encoder placeholder,
que sozinho (via texto) já não é bom o suficiente pra essa tarefa.

In [10]:
grupos = motor.clusters_por_natureza("furto", threshold=0.3)

print(f"{len(grupos)} grupo(s) encontrado(s):")
for i, g in enumerate(grupos):
    print(f"  grupo {i}: {g}")

1 grupo(s) encontrado(s):
  grupo 0: ['BO-001', 'BO-005', 'BO-006', 'BO-007']


## 6. Caso de uso 3 — "Essa pessoa tem mais de um BO?"

Esperado: buscar por "Joao da Silva" deve encontrar tanto `BO-003`
quanto `BO-004`, mesmo com a diferença de espaçamento/grafia
("JOAO  DA  SILVA" com espaço duplo).

In [11]:
candidatos = motor.bos_da_pessoa("Joao da Silva", limiar=0.85)

for c in candidatos:
    print(f"  {c['bo_id']}: score={c['score']:.3f} | natureza={c['natureza']}")

  BO-003: score=1.000 | natureza=estelionato
  BO-004: score=1.000 | natureza=estelionato


## 7. Persistência — salvar e recarregar o índice

Simula o ciclo real de produção: salvar o estado, "reiniciar o serviço"
(nova instância) e recarregar, conferindo que os resultados batem.

In [12]:
motor.salvar("teste_indice")
print("Salvo em disco:", [f for f in os.listdir(".") if f.startswith("teste_indice")])

motor_recarregado = MotorDeLinkage.carregar("teste_indice", encoder, geo_lookup)
print(f"\nBOs recarregados: {len(motor_recarregado.ids)}")

# confirma que o resultado do caso de uso 1 é o mesmo antes/depois de recarregar
score_antes = motor.score_par("BO-001", "BO-002")
score_depois = motor_recarregado.score_par("BO-001", "BO-002")
print(f"score BO-001 x BO-002 antes de salvar:  {score_antes:.4f}")
print(f"score BO-001 x BO-002 depois de recarregar: {score_depois:.4f}")
assert abs(score_antes - score_depois) < 1e-9, "Divergência após recarregar!"
print("OK -- resultado idêntico antes e depois de salvar/recarregar.")

Salvo em disco: ['teste_indice_classificador.joblib', 'teste_indice_embeddings.npy', 'teste_indice_indice.json']

BOs recarregados: 8
score BO-001 x BO-002 antes de salvar:  0.6430
score BO-001 x BO-002 depois de recarregar: 0.6430
OK -- resultado idêntico antes e depois de salvar/recarregar.


## 8. Adicionando um BO novo ao índice já existente

Simula o fluxo real: o índice já está carregado (de `carregar()`), chega
um BO novo, você só chama `adicionar_bo` normalmente.

In [13]:
bo_novo = {
    "id": "BO-009",
    "texto": "Mais um furto de veiculo, placa ABC1D23, na regiao central.",
    "data_fato": "15/07/2026",
    "municipio": "RECIFE",
    "natureza": "furto",
    "entidades": [{"texto": "ABC1D23", "tipo": "PLACA"}],
}

motor_recarregado.adicionar_bo(bo_novo, entidades=bo_novo["entidades"])

acima, _ = motor_recarregado.bos_conectados("BO-009", k=5, threshold=0.3)
print("BO-009 conecta com:")
for r in acima:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

BO-009 conecta com:
  BO-001: score=0.674 | motivo: mesmo(a) PLACA
  BO-002: score=0.633 | motivo: mesmo(a) PLACA
  BO-007: score=0.327 | motivo: similaridade semântica
  BO-008: score=0.304 | motivo: similaridade semântica


## 9. Limpeza (opcional)

Remove os arquivos de teste gerados na célula 7.

In [ ]:
for f in os.listdir("."):
    if f.startswith("teste_indice"):
        os.remove(f)
        print("removido:", f)